# HI5312 Excel Tutorial Guide

## Excel for Health Informatics in Codespaces

This notebook supports the 60-minute Excel tutorial. It is **not a replacement for Excel**. Instead, it helps you:

- locate the course practice files in Codespaces;
- inspect CSV and XLSX files before opening them in Excel;
- understand table grain and data types;
- validate the logic of Excel formulas;
- see the Python equivalent of an Excel VLOOKUP using `zip-sdi.xlsx`.

You will use Excel for the hands-on parts. Use this notebook as a guide and validation companion.

## 1. Locate the course data files

The course data files are stored in the `data` folder of the Codespaces repository. This cell searches likely locations.

In [4]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/workspaces/hi5304-notebooks/data")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory not found: {DATA_DIR}"
    )

files_to_check = [
    "patients.csv",
    "bp_readings.csv",
    "medications.csv",
    "cardio1.csv",
    "cardio10.csv",
    "zip-sdi.csv",
]

print("Data folder:", DATA_DIR)
print()

for file_name in files_to_check:
    path = DATA_DIR / file_name
    print(f"{file_name:20} {'FOUND' if path.exists() else 'NOT FOUND'}")

Data folder: /workspaces/hi5304-notebooks/data

patients.csv         FOUND
bp_readings.csv      FOUND
medications.csv      FOUND
cardio1.csv          FOUND
cardio10.csv         FOUND
zip-sdi.csv          FOUND


## 2. Inspect headers before opening in Excel

Before editing data in Excel, inspect the file structure. Look at the column names and ask: **what does one row represent?**

In [6]:
for file_name in [
    'patients.csv',
    'bp_readings.csv',
    'medications.csv'
]:
    df = pd.read_csv(DATA_DIR / file_name)

    print('\n' + file_name)
    print('Rows, columns:', df.shape)
    print('Columns:', list(df.columns))

    display(df.head())


patients.csv
Rows, columns: (5, 8)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'sex', 'race', 'ethnicity', 'zip_code']


,patient_id,first_name,last_name,date_of_birth,sex,race,ethnicity,zip_code
0,1001,John,Doe,4/12/1968,M,White,Non-Hispanic,75201
1,1002,Maria,Lopez,9/30/1975,F,Hispanic,Hispanic,75204
2,1003,James,Smith,1/18/1982,M,Black,Non-Hispanic,75080
3,1004,Linda,Chen,6/5/1990,F,Asian,Non-Hispanic,75024
4,1005,Robert,Johnson,11/22/1959,M,White,Non-Hispanic,75230



bp_readings.csv
Rows, columns: (9, 7)
Columns: ['bp_id', 'patient_id', 'reading_date', 'systolic', 'diastolic', 'heart_rate', 'source']


,bp_id,patient_id,reading_date,systolic,diastolic,heart_rate,source
0,1,1001,9/1/2024,148,92,78,Clinic
1,2,1001,10/1/2024,142,88,76,Home
2,3,1001,11/1/2024,136,84,74,Home
3,4,1002,9/15/2024,132,82,70,Clinic
4,5,1002,10/15/2024,128,80,68,Home



medications.csv
Rows, columns: (5, 7)
Columns: ['med_id', 'patient_id', 'medication_name', 'dose', 'frequency', 'start_date', 'indication']


,med_id,patient_id,medication_name,dose,frequency,start_date,indication
0,1,1001,Lisinopril,10 mg,Once daily,9/1/2024,Hypertension
1,2,1002,Amlodipine,5 mg,Once daily,8/20/2024,Hypertension
2,3,1003,Metoprolol,50 mg,Twice daily,9/10/2024,Hypertension
3,4,1003,Atorvastatin,20 mg,Once daily,9/10/2024,Hyperlipidemia
4,5,1005,Losartan,50 mg,Once daily,9/5/2024,Hypertension


## 3. Grain: what does one row represent?

Complete this table as part of your practice:

| File | What one row represents | Key identifier | Notes |
|---|---|---|---|
| `patients.csv` | One patient | `patient_id` | Patient-level demographics |
| `bp_readings.csv` | One blood pressure reading | `bp_id` | Repeated readings per patient |
| `medications.csv` | One medication record | `med_id` | One patient may have multiple medications |
| `zip-sdi.xlsx` | One ZIP/ZCTA area | `ZCTA5_FIPS` | Lookup table for SDI |

## 4. Check data types

Excel sometimes changes data types automatically. In health informatics, this matters. ZIP codes and IDs may look numeric but often behave like identifiers.

In [11]:
patients = pd.read_csv(
    DATA_DIR / 'patients.csv',
    dtype={'zip_code': str}
)

bp = pd.read_csv(DATA_DIR / 'bp_readings.csv')

meds = pd.read_csv(DATA_DIR / 'medications.csv')

print('patients.csv data types')
print(patients.dtypes)

print('\nbp_readings.csv data types')
print(bp.dtypes)

print('\nmedications.csv data types')
print(meds.dtypes)

patients.csv data types
patient_id       int64
first_name         str
last_name          str
date_of_birth      str
sex                str
race               str
ethnicity          str
zip_code           str
dtype: object

bp_readings.csv data types
bp_id           int64
patient_id      int64
reading_date      str
systolic        int64
diastolic       int64
heart_rate      int64
source            str
dtype: object

medications.csv data types
med_id             int64
patient_id         int64
medication_name      str
dose                 str
frequency            str
start_date           str
indication           str
dtype: object


## 5. Excel Table practice

In Excel:

1. Open `patients.csv`.
2. Click inside the dataset.
3. Press **Ctrl+T** or select **Insert → Table**.
4. Confirm **My table has headers**.
5. Name the table `Patients`.
6. Repeat for `bp_readings.csv` and `medications.csv`.

Why this matters: Excel Tables preserve row integrity, make filters easier, and allow structured formulas.

## 6. Formula examples to try in Excel

In an Excel Table, try formulas like these:

```excel
=DATEDIF([@[date_of_birth]], TODAY(), "Y")
```

```excel
=IF(AND([@[systolic]]<140,[@[diastolic]]<90),"Controlled","Not controlled")
```

```excel
=[@[systolic]]-[@[diastolic]]
```

These create derived variables: age, blood pressure status, and pulse pressure.

## 7. Validate the blood pressure status logic in Python

This is the Python equivalent of the Excel `IF(AND(...))` formula.

In [ ]:
bp['reading_date'] = pd.to_datetime(bp['reading_date'])
bp['bp_status'] = bp.apply(
    lambda row: 'Controlled' if row['systolic'] < 140 and row['diastolic'] < 90 else 'Not controlled',
    axis=1
)

bp[['patient_id', 'reading_date', 'systolic', 'diastolic', 'bp_status']]

## 8. VLOOKUP with zip-sdi.xlsx

Goal: add SDI information to the patient table using ZIP code.

In Excel, after opening `patients.csv` and `zip-sdi.xlsx`, create a new column in the patient table called `SDI_score` and use:

```excel
=VLOOKUP([@[zip_code]], 'zip-sdi'!$A:$R, 3, FALSE)
```

To return the `sdi` variable instead, use the column number where `sdi` appears in the lookup range. In the supplied file, `sdi` is column 11 in the A:R range:

```excel
=VLOOKUP([@[zip_code]], 'zip-sdi'!$A:$R, 11, FALSE)
```

Use `FALSE` for exact match.

## 9. Python equivalent of VLOOKUP

In Python, a VLOOKUP is usually done with a merge. This cell shows the same logic.

In [14]:
zip_sdi = pd.read_csv(
    DATA_DIR / 'zip-sdi.csv',
    dtype={'ZCTA5_FIPS': str}
)

# Standardize ZIP codes as 5-character strings
patients['zip_code'] = (
    patients['zip_code']
    .astype(str)
    .str.zfill(5)
)

zip_sdi['ZCTA5_FIPS'] = (
    zip_sdi['ZCTA5_FIPS']
    .astype(str)
    .str.zfill(5)
)

# Merge patient data with SDI data using ZIP code
patients_with_sdi = patients.merge(
    zip_sdi[
        ['ZCTA5_FIPS', 'SDI_score', 'sdi']
    ],
    left_on='zip_code',
    right_on='ZCTA5_FIPS',
    how='left'
)

patients_with_sdi[
    ['patient_id', 'zip_code', 'SDI_score', 'sdi']
]

,patient_id,zip_code,SDI_score,sdi
0,1001,75201,55.0,0.142212
1,1002,75204,71.0,0.597831
2,1003,75080,60.0,0.264314
3,1004,75024,23.0,-0.832327
4,1005,75230,24.0,-0.796525


## 10. Validate the lookup

After any lookup, ask:

- How many rows matched?
- How many did not match?
- Are ZIP codes stored the same way in both tables?
- Are there duplicate lookup keys?
- Did the formula return the intended column?

In [15]:
print('Rows in patients:', len(patients))
print('Rows after merge:', len(patients_with_sdi))
print('Missing SDI_score:', patients_with_sdi['SDI_score'].isna().sum())
print('Duplicate ZIP keys in zip_sdi:', zip_sdi['ZCTA5_FIPS'].duplicated().sum())

Rows in patients: 5
Rows after merge: 5
Missing SDI_score: 0
Duplicate ZIP keys in zip_sdi: 0


## 11. Create a patient-level analytic file conceptually

For the mini-lab, you can build a patient-level file with:

- demographics from `patients.csv`;
- latest blood pressure status from `bp_readings.csv`;
- hypertension medication indicator from `medications.csv`;
- SDI variables from `zip-sdi.xlsx`.

This demonstrates grain, joins/lookups, derived variables, and export.

In [17]:
# Convert reading_date to a true datetime variable
bp['reading_date'] = pd.to_datetime(bp['reading_date'])

# Find the most recent blood pressure for each patient
latest_bp = (
    bp.sort_values('reading_date')
      .groupby('patient_id')
      .tail(1)
)

# Keep the variables that actually exist in bp_readings.csv
latest_bp = latest_bp[
    [
        'patient_id',
        'reading_date',
        'systolic',
        'diastolic',
        'heart_rate',
        'source'
    ]
]

# Identify medications with hypertension as the indication
htn_meds = meds[
    meds['indication']
    .str.contains('Hypertension', case=False, na=False)
]

# Count hypertension medications for each patient
htn_med_indicator = (
    htn_meds.groupby('patient_id')
    .size()
    .rename('n_hypertension_meds')
    .reset_index()
)

# Merge patient + SDI data with the latest blood pressure
analytic = patients_with_sdi.merge(
    latest_bp,
    on='patient_id',
    how='left'
)

# Add hypertension medication information
analytic = analytic.merge(
    htn_med_indicator,
    on='patient_id',
    how='left'
)

# Patients without a hypertension medication receive a count of 0
analytic['n_hypertension_meds'] = (
    analytic['n_hypertension_meds']
    .fillna(0)
    .astype(int)
)

# Create a simple True/False medication indicator
analytic['has_hypertension_med'] = (
    analytic['n_hypertension_meds'] > 0
)

analytic.head()

,patient_id,first_name,last_name,date_of_birth,sex,race,ethnicity,zip_code,ZCTA5_FIPS,SDI_score,sdi,reading_date,systolic,diastolic,heart_rate,source,n_hypertension_meds,has_hypertension_med
0,1001,John,Doe,4/12/1968,M,White,Non-Hispanic,75201,75201,55.0,0.142212,2024-11-01,136,84,74,Home,1,True
1,1002,Maria,Lopez,9/30/1975,F,Hispanic,Hispanic,75204,75204,71.0,0.597831,2024-10-15,128,80,68,Home,1,True
2,1003,James,Smith,1/18/1982,M,Black,Non-Hispanic,75080,75080,60.0,0.264314,2024-10-10,146,94,80,Home,1,True
3,1004,Linda,Chen,6/5/1990,F,Asian,Non-Hispanic,75024,75024,23.0,-0.832327,2024-09-20,118,76,66,Clinic,0,False
4,1005,Robert,Johnson,11/22/1959,M,White,Non-Hispanic,75230,75230,24.0,-0.796525,2024-09-05,140,90,75,Clinic,1,True


## 12. Export a clean CSV for the next tool

When moving from Excel to SQL or Python, use a simple rectangular CSV:

- one header row;
- no merged cells;
- no totals rows mixed into the data;
- no hidden filters;
- clear variable names;
- one row per unit of analysis.

In [ ]:
output_path = DATA_DIR / 'hi5312_excel_practice_patient_level.csv'
analytic.to_csv(output_path, index=False)
print('Saved:', output_path)

# Key takeaways

- Excel is useful for inspecting, formatting, and quality-checking data.
- Always identify the grain of the data before summarizing or joining.
- IDs and ZIP codes are identifiers, not measurements.
- VLOOKUP is powerful but must be validated.
- Exact match matters.
- Excel work should prepare data for SQL, Python, and Tableau, not replace those tools.